# Week 3 exercise answers: selecting and grouping baby-name data

These are worked solutions. Other clear approaches can also be correct.

Variables containing DataFrames end in `_df`; important Series variables end in
`_series`. These suffixes are labels for readers, not requirements imposed by Python.

## Setup

**Question:** Load the US regional and Scottish baby-name CSV files as DataFrames.

In [1]:
import pandas as pd
from IPython.display import display

# The _df suffix reminds us that each variable contains a DataFrame.
usa_df = pd.read_csv("us_babynames_by_region.csv")
scotland_df = pd.read_csv("scotland_babynames.csv")

## Inspect the US DataFrame

**Question:** Display the first five rows, shape, column names, and data types.
Describe what one row represents.

One row contains the recorded count for one name, sex, year, and US Census region.

In [2]:
# head() lets us see the table's structure and a few example observations.
display(usa_df.head())

# shape reports (number of rows, number of columns).
print("Shape:", usa_df.shape)

# columns and dtypes tell us which variables exist and what values they contain.
print("Columns:", list(usa_df.columns))
display(usa_df.dtypes)

,Region,Sex,Year,Name,Count
0,Midwest,F,1910,Mary,5024
1,Midwest,F,1910,Helen,3747
2,Midwest,F,1910,Dorothy,2677
3,Midwest,F,1910,Margaret,2502
4,Midwest,F,1910,Ruth,2377


Shape: (1628156, 5)
Columns: ['Region', 'Sex', 'Year', 'Name', 'Count']


Region      str
Sex         str
Year      int64
Name        str
Count     int64
dtype: object

## Series or DataFrame?

**Question:** Select `Name` once using the string `"Name"` and once using the
one-item list `["Name"]`. Compare their types and shapes. Why do they differ?

A single column label returns a one-dimensional Series. A list asks pandas to
construct a table, so even a one-item list returns a two-dimensional DataFrame.

In [3]:
# A single string asks for the column itself, producing a Series.
name_series = usa_df["Name"]

# A list asks for a smaller table, producing a one-column DataFrame.
name_df = usa_df[["Name"]]

# The Series shape has one dimension; the DataFrame shape has two.
print(type(name_series), name_series.shape)
print(type(name_df), name_df.shape)

<class 'pandas.Series'> (1628156,)
<class 'pandas.DataFrame'> (1628156, 1)


## Select and sort

**Question:** Find the ten most frequently recorded girls' names in the Northeast
in 2025. Which is first, how much larger is it than tenth place, and do the counts
drop steeply or remain fairly similar?

In [4]:
# Ask the three row-selection questions separately so each can be inspected.
select_region = usa_df["Region"] == "Northeast"
select_female = usa_df["Sex"] == "F"
select_year = usa_df["Year"] == 2025

# A row is retained only when all three conditions are True.
select_rows = select_region & select_female & select_year
selected_df = usa_df.loc[select_rows, ["Name", "Count"]]

# Reuse selected_df: sorting does not create a conceptually different dataset.
selected_df = selected_df.sort_values("Count", ascending=False)
selected_df.head(10)

,Name,Count
692268,Charlotte,2341
692269,Sophia,2257
692270,Olivia,2211
692271,Emma,2170
692272,Mia,1956
692273,Amelia,1952
692274,Isabella,1798
692275,Ava,1411
692276,Sofia,1353
692277,Nora,1318


**Interpretation:** Charlotte is first with 2,341 recorded babies; Nora is tenth
with 1,318. The difference is 1,023. Counts decline across the list, with a larger
drop near the bottom rather than one name completely dominating the table.

## Total recorded count by region

**Question:** For 2025, calculate the total recorded count in every US region.
Which region is largest, which is smallest, and approximately how many times larger
is the largest?

In [5]:
# This year subset will be reused in several later questions, so it deserves a name.
select_2025 = usa_df["Year"] == 2025
usa_2025_df = usa_df.loc[select_2025]

# groupby creates one group per region; sum() adds Count within each group.
grouped_by_region = usa_2025_df.groupby("Region", as_index=False)
totals_by_region_df = grouped_by_region[["Count"]].sum()

totals_by_region_df

,Region,Count
0,Midwest,551280
1,Northeast,435748
2,South,1142958
3,West,639768


**Interpretation:** The South has the largest recorded total (1,142,958) and the
Northeast the smallest (435,748). The South's total is roughly 2.6 times the
Northeast's.

## The same data, a different grouping question

**Question:** Group the 2025 US data by region and sex, then count the number of
different recorded names. Which region has the most? Does each region contain
more girls' names or boys' names?

In [6]:
# usa_2025_df already contains only 2025, so we can reuse it.
# Supplying two column names asks pandas to form one group for each
# Region-and-Sex combination: four regions multiplied by two sexes.
grouped_by_region_and_sex = usa_2025_df.groupby(["Region", "Sex"], as_index=False)

# Select Name because that is the variable whose different values we want to count.
# nunique() means "number of unique values" and produces one row per group.
different_names_df = grouped_by_region_and_sex[["Name"]].nunique()

different_names_df

,Region,Sex,Name
0,Midwest,F,2423
1,Midwest,M,2114
2,Northeast,F,2745
3,Northeast,M,2373
4,South,F,4264
5,South,M,3403
6,West,F,3617
7,West,M,2930


**Interpretation:** The South has the most different recorded names for
both sexes. Every region contains more different girls' names than boys' names,
so the same pattern appears in all four regions.

Remember that the source data omit some very rare names, so “different recorded
names” means names that are present in these files.

## Scotland in 2025

**Question:** Find Scotland's ten leading girls' and boys' names in 2025. What
are the two leaders, and which has the larger count?

In [7]:
# Looking at a few rows first confirms the column names and layout.
display(scotland_df.head())

# Ask whether each row comes from 2025, then retain only the True rows.
select_2025 = scotland_df["Year"] == 2025
scotland_2025_df = scotland_df.loc[select_2025]

# Sort once because both the girls' and boys' displays need the same ordering.
scotland_2025_df = scotland_2025_df.sort_values("Count", ascending=False)

# The Boolean Series identifies girls' rows in the 2025 DataFrame.
# .loc uses it to retain those rows and the two useful display columns.
select_girls = scotland_2025_df["Sex"] == "F"
display(scotland_2025_df.loc[select_girls, ["Name", "Count"]].head(10))

# Repeat the same selection pattern for boys.
select_boys = scotland_2025_df["Sex"] == "M"
display(scotland_2025_df.loc[select_boys, ["Name", "Count"]].head(10))

,Sex,Year,Name,Count,Rank
0,M,1974,Aaron,17,139
1,M,1974,Abdul,5,248
2,M,1974,Adam,75,78
3,M,1974,Adrian,46,92
4,M,1974,Aftab,3,297


,Name,Count
73307,Freya,268
73308,Isla,253
73309,Olivia,251
73310,Amelia,204
73311,Grace,200
73312,Emily,196
73313,Millie,195
73314,Lily,193
73315,Sophia,181
73316,Rosie,177


,Name,Count
72349,Noah,318
72350,Luca,302
72351,Rory,301
72352,Muhammad,291
72353,Oliver,269
72354,Theo,243
72355,Leo,236
72356,Archie,222
72357,Finlay,216
72358,Harris,211


**Interpretation:** Freya leads the girls' table with 268 recorded
babies; Noah leads the boys' table with 318. Noah's count is 50 larger. You may
also notice the cluster of short boys' names such as Luca, Rory, Theo, and Leo.

## Scotland versus the US Northeast

**Question:** Display the 2025 girls' top tens for the US Northeast and Scotland,
then repeat for boys. Which names occur in both corresponding lists?

In [8]:
# ----- Girls in the US Northeast -----
# Build one simple Boolean question at a time. Each result can be inspected.
select_region = usa_df["Region"] == "Northeast"
select_year = usa_df["Year"] == 2025
select_female = usa_df["Sex"] == "F"

# & means all three conditions must be True for a row to be retained.
select_rows = select_region & select_year & select_female
selected_df = usa_df.loc[select_rows, ["Name", "Count"]]

# Reuse selected_df because sorting changes its order, not what it represents.
selected_df = selected_df.sort_values("Count", ascending=False)
print("US Northeast — girls")
display(selected_df.head(10))

# ----- Girls in Scotland -----
# Scotland has no Region column, so this selection only needs year and sex.
select_year = scotland_df["Year"] == 2025
select_female = scotland_df["Sex"] == "F"
select_rows = select_year & select_female
selected_df = scotland_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("Scotland — girls")
display(selected_df.head(10))

US Northeast — girls


,Name,Count
692268,Charlotte,2341
692269,Sophia,2257
692270,Olivia,2211
692271,Emma,2170
692272,Mia,1956
692273,Amelia,1952
692274,Isabella,1798
692275,Ava,1411
692276,Sofia,1353
692277,Nora,1318


Scotland — girls


,Name,Count
73307,Freya,268
73308,Isla,253
73309,Olivia,251
73310,Amelia,204
73311,Grace,200
73312,Emily,196
73313,Millie,195
73314,Lily,193
73315,Sophia,181
73316,Rosie,177


In [9]:
# ----- Boys in the US Northeast -----
# Repeat the same structure, changing only the sex question from F to M.
select_region = usa_df["Region"] == "Northeast"
select_year = usa_df["Year"] == 2025
select_male = usa_df["Sex"] == "M"
select_rows = select_region & select_year & select_male
selected_df = usa_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("US Northeast — boys")
display(selected_df.head(10))

# ----- Boys in Scotland -----
select_year = scotland_df["Year"] == 2025
select_male = scotland_df["Sex"] == "M"
select_rows = select_year & select_male
selected_df = scotland_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("Scotland — boys")
display(selected_df.head(10))

US Northeast — boys


,Name,Count
695013,Noah,3803
695014,Liam,3558
695015,Theodore,2435
695016,Oliver,2271
695017,James,2084
695018,Lucas,2044
695019,Henry,2017
695020,Jack,1929
695021,Luca,1919
695022,Benjamin,1857


Scotland — boys


,Name,Count
72349,Noah,318
72350,Luca,302
72351,Rory,301
72352,Muhammad,291
72353,Oliver,269
72354,Theo,243
72355,Leo,236
72356,Archie,222
72357,Finlay,216
72358,Harris,211


**Interpretation:** The shared girls' names are `Olivia`, `Amelia`,
and `Sophia`. The shared boys' names are `Noah`, `Luca`, and `Oliver`. Most names
in each pair of lists are different, but there is still some overlap.


## Stretch goal: three naming generations

**Question:** Compare the US South's top ten girls' and boys' names in 1980,
2000, and 2025. Which names persist, which sex appears more stable, and which
period seems to show the greatest change?

In [10]:
# Select the South once because all six tables come from this region.
select_south = usa_df["Region"] == "South"
south_df = usa_df.loc[select_south]

# ----- Girls -----
# For each year, combine a year question with the same female selection.
select_female = south_df["Sex"] == "F"

select_year = south_df["Year"] == 1980
select_rows = select_year & select_female
selected_df = south_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("South — girls — 1980")
display(selected_df.head(10))


select_year = south_df["Year"] == 2000
select_rows = select_year & select_female
selected_df = south_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("South — girls — 2000")
display(selected_df.head(10))

select_year = south_df["Year"] == 2025
select_rows = select_year & select_female
selected_df = south_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("South — girls — 2025")
display(selected_df.head(10))

South — girls — 1980


,Name,Count
901935,Jennifer,18729
901936,Amanda,15112
901937,Jessica,11110
901938,Melissa,9354
901939,Kimberly,7566
901940,Crystal,6810
901941,Tiffany,6719
901942,Amy,6468
901943,Heather,6450
901944,Stephanie,6370


South — girls — 2000


,Name,Count
997144,Hannah,9213
997145,Emily,8777
997146,Madison,8108
997147,Sarah,6775
997148,Alexis,6730
997149,Ashley,6428
997150,Taylor,6180
997151,Elizabeth,5352
997152,Kayla,5333
997153,Lauren,5296


South — girls — 2025


,Name,Count
1176712,Olivia,5449
1176713,Amelia,5338
1176714,Charlotte,5298
1176715,Emma,5298
1176716,Sophia,4777
1176717,Isabella,4658
1176718,Mia,4456
1176719,Eliana,3521
1176720,Evelyn,3464
1176721,Sofia,3401


In [11]:
# ----- Boys -----
# The structure is deliberately the same, which makes the changed parts obvious.
select_male = south_df["Sex"] == "M"

select_year = south_df["Year"] == 1980
select_rows = select_year & select_male
selected_df = south_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("South — boys — 1980")
display(selected_df.head(10))

select_year = south_df["Year"] == 2000
select_rows = select_year & select_male
selected_df = south_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("South — boys — 2000")
display(selected_df.head(10))

select_year = south_df["Year"] == 2025
select_rows = select_year & select_male
selected_df = south_df.loc[select_rows, ["Name", "Count"]]
selected_df = selected_df.sort_values("Count", ascending=False)
print("South — boys — 2025")
display(selected_df.head(10))

South — boys — 1980


,Name,Count
904309,Michael,20652
904310,Christopher,18227
904311,James,16528
904312,Jason,16518
904313,David,13531
904314,John,13039
904315,Joshua,12207
904316,Robert,12045
904317,William,11124
904318,Matthew,9539


South — boys — 2000


,Name,Count
1000688,Jacob,12409
1000689,Michael,11089
1000690,Joshua,10904
1000691,Christopher,10287
1000692,William,9914
1000693,Matthew,9577
1000694,John,8084
1000695,James,8009
1000696,Brandon,7695
1000697,Nicholas,7645


South — boys — 2025


,Name,Count
1180976,Liam,9045
1180977,Noah,8524
1180978,Oliver,5575
1180979,Elijah,5270
1180980,James,5051
1180981,William,4717
1180982,Mateo,4583
1180983,Henry,4332
1180984,Theodore,4237
1180985,Lucas,4190


**Interpretation:** The boys' lists appear more stable. Seven boys'
names from the 1980 top ten are still present in 2000, and `James` and `William`
remain in the top ten in all three years. None of the girls' top-ten names carries
over between these particular years. The girls' lists therefore show complete
turnover in both intervals; for boys, the larger change occurs from 2000 to 2025.

This is a descriptive pattern in the South's top tens, not a universal rule about
all names or populations.

## Stretch debugging: a wrong answer

This code claims to count the number of different names in every US region in
2025. It runs, but the answer is wrong. Come up with a sanity check of the dataset to show the output is wrong, identify what's wrong, then fix the problem.  

**Interpretation:** After fixing the column, which region has the greatest number
of different recorded names?

In [12]:
# Select 2025 and group it by region.
select_2025 = usa_df["Year"] == 2025
usa_2025_df = usa_df.loc[select_2025]
grouped_by_region = usa_2025_df.groupby("Region")

# This runs, but nunique() is applied to Count rather than Name.
wrong_answer_df = grouped_by_region[["Count"]].nunique()
wrong_answer_df

,Count
Region,
Midwest,635
Northeast,541
South,884
West,671


Hmmm.  That seems like it could be a bit low for all the different names in a region.  Let's see how many unique names there are. 

In [13]:
usa_2025_df['Name'].nunique()

9138

Thats 10x more than the counts above something must be wrong.  Let's look closely at what is being counted. 

The code counts different numerical count values, not different names. The method
is valid; the selected column is wrong.

In [14]:
# Select Name before nunique() so the method answers the intended question.
corrected_answer_df = grouped_by_region[["Name"]].nunique()
corrected_answer_df

,Name
Region,
Midwest,4314
Northeast,4903
South,7199
West,6213


**Interpretation:** The South has the greatest number of different recorded names
in 2025.